In [ ]:
import os
import json
import io

import re
from datetime import datetime
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

import numpy as np
import pandas as pd

# Connect to Google Drive
import gspread
import gspread_dataframe
from google.oauth2.service_account import Credentials
from google.oauth2 import service_account
from gspread_dataframe import set_with_dataframe
from gspread_dataframe import get_as_dataframe


In [ ]:
# 1. Fetch credentials from environment variable
creds_env = os.environ.get("GDRIVE_CREDENTIALS_KC")

if not creds_env:
    raise ValueError("Environment variable 'GDRIVE_CREDENTIALS' was not found.")

creds_json = json.loads(creds_env)

# 2. Define required scopes
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

# 3. Authenticate service account
creds = service_account.Credentials.from_service_account_info(
    creds_json, scopes=scopes
)

# 4. Initialize Google API clients
drive_service = build("drive", "v3", credentials=creds)
sheets_service = build("sheets", "v4", credentials=creds)

gc = gspread.authorize(creds)

print("Google Drive and Sheets services successfully initialized.")

Mounted at /content/drive


In [20]:
# ID of the Drive folder that holds the Sprinklr Paid exports.
# Get it from the folder URL: drive.google.com/drive/folders/<THIS_IS_THE_ID>
DRIVE_FOLDER_ID = "1k8NDz3qxQ9ffZzkS2EsWT1tNSk3ghklU"

# File name pattern: DDMMYYYY.xlsx (e.g. 31072026.xlsx)
FILENAME_PATTERN = re.compile(r"^(\d{2})(\d{2})(\d{4})\.xlsx$")

In [21]:
# Build the Drive service reusing the service-account creds already created above
drive_service = build("drive", "v3", credentials=creds)

def find_latest_file(drive_service):
    query = f"'{DRIVE_FOLDER_ID}' in parents and trashed = false"
    results = drive_service.files().list(
        q=query,
        fields="files(id, name)",
        pageSize=1000,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
        corpora="allDrives",
    ).execute()

    files = results.get("files", [])
    print(f"[DEBUG] Archivos visibles en la carpeta {DRIVE_FOLDER_ID}: {len(files)}")
    for f in files:
        print(f"[DEBUG]  - {f['name']} (id={f['id']})")

    candidates = []
    for f in files:
        m = FILENAME_PATTERN.match(f["name"])
        if not m:
            continue
        dd, mm, yyyy = m.groups()
        try:
            file_date = datetime(int(yyyy), int(mm), int(dd))
        except ValueError:
            continue
        candidates.append((file_date, f))

    if not files:
        raise RuntimeError(
            "No se retornó ningún archivo para esa carpeta. Probablemente la service "
            "account no tiene acceso, o el ID de la carpeta está mal."
        )
    if not candidates:
        raise RuntimeError(
            "No se encontró ningún archivo con el patrón DDMMYYYY.xlsx en la carpeta. "
            "Revisá la lista [DEBUG] de arriba para ver los nombres reales."
        )

    candidates.sort(key=lambda x: x[0])
    latest_date, latest_file = candidates[-1]
    print(f"Archivo más reciente encontrado: {latest_file['name']} (fecha {latest_date.date()})")
    return latest_file["id"], latest_file["name"]

def download_file(drive_service, file_id, local_path):
    request = drive_service.files().get_media(fileId=file_id, supportsAllDrives=True)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
    with open(local_path, "wb") as f:
        f.write(fh.getvalue())
    print(f"Archivo descargado en: {local_path}")

# Find the latest file, download it, and open it
file_id, file_name = find_latest_file(drive_service)
file_path = f"/tmp/{file_name}"
download_file(drive_service, file_id, file_path)

sprinklr_paid_new = pd.read_excel(file_path, sheet_name='WinClap_Paid Media', header=2)
sprinklr_paid_new = sprinklr_paid_new.fillna(0)

[DEBUG] Archivos visibles en la carpeta 1k8NDz3qxQ9ffZzkS2EsWT1tNSk3ghklU: 38
[DEBUG]  - 31072026.xlsx (id=1yeSt-ncuAYIbzUvJRR3xKo6iE7kAtCms)
[DEBUG]  - 04082026.xlsx (id=15TVzoofX7SqP2qNE_HOgzdATJX5HNgWc)
[DEBUG]  - 03082026.xlsx (id=1fIz4AFrwt6GnlZZYaAWdsOkvJRw-uHsR)
[DEBUG]  - 30072026.xlsx (id=1c_S9cZRda1wrZug6TluMWRKSrZPovIqj)
[DEBUG]  - backfill_6months_jan26-jul26.xlsx (id=1i8uwHxPyZJTaaS4Zb5JTkVS4Cc8CY4IR)
[DEBUG]  - 27072026.xlsx (id=1_PushSIvVEIYnYQOhYwUwqSzw94M_Ox-)
[DEBUG]  - 29072026.xlsx (id=1NRjUijgg1yh3fHTM0Csi2evsVcv77fTZ)
[DEBUG]  - 25072026.xlsx (id=1MKCELrqPLqroCfkVggyIimKXGyFG3_-_)
[DEBUG]  - 23072026.xlsx (id=1Z0_ObvcN56oSBLBwFcH7_DI8bxmqZI3j)
[DEBUG]  - 20072026.xlsx (id=11j42OkVux0jPR0cHa7J-3gl4r8gSUGkX)
[DEBUG]  - Copia de 20072026.xlsx (id=1zS-3YotpBmJYv15xAIl8thkpjuxG452P)
[DEBUG]  - 12062026.xlsx (id=1sl6SgPJpg2U3bWfZP-f4aOWrgpu15qVs)
[DEBUG]  - 18072026.xlsx (id=1wgj5wi0l5TOt78HyZU_saRl6iawD2rhU)
[DEBUG]  - 17072026.xlsx (id=1AWKs5L6CAzKE-rsz0wiJPReI8JdLnZZ

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [23]:
# Extract organic id from 'Ad Variant Name'
sprinklr_paid_new['Organic_ID'] = sprinklr_paid_new['Ad Variant Name'].str.split('_').str[4]

In [24]:
# Keep only the rows that match with organic posts and influencer posts (inner join with the final organic table) -> we do this so the paid data file doesn't become huge due to unnecesary data (ads that are not organic boosting)

# Open organic posts file
organic_posts = gc.open_by_key('1FnauIqLuTe1c2N8Z-HQPy8wambQzBhpbLJY24JMCMNY')
organic_posts = organic_posts.worksheet('Hoja 1')
organic_posts = get_as_dataframe(organic_posts)
organic_posts = organic_posts['Organic_ID']

# Open influencers posts file
influencer_posts = gc.open_by_key('1QwqDvUu5SAt6PHKBZWWOATkqzE58pN_XDnzo6LMiYOI')
influencer_posts = influencer_posts.worksheet('Sheet1')
influencer_posts = get_as_dataframe(influencer_posts)
influencer_posts = influencer_posts['Organic_ID']

# Append organic and influencers
posts_filter_list = pd.concat([organic_posts, influencer_posts], ignore_index=True)


In [25]:
# Keep only the rows that match with organic posts and influencer posts (inner join with posts_filter_list) -> we do this so the paid data file doesn't become huge due to unnecesary data (ads that are not organic boosting)

# Filter the rows in sprinklr_paid_new that match with Organic_ID in organic posts
sprinklr_paid_new = sprinklr_paid_new[sprinklr_paid_new['Organic_ID'].isin(posts_filter_list)].reset_index(drop=True)
sprinklr_paid_new = sprinklr_paid_new.dropna(subset=['Organic_ID'])

sprinklr_paid_new

,Date,Ad Account,Social Network,Paid Initiative Name,Ad Variant Name,Ad Variant Id,Ad Variant,Title,Body,Image URL,...,Facebook Reactions (SUM),Facebook Post Saves (SUM),Neutral Sentiment Count (Paid + Organic) (AVG),Positive Sentiment Count (Paid + Organic) (AVG),Negative Sentiment Count (Paid + Organic) (AVG),Facebook Avg. Duration of Video Played (SUM),Ad Post Id,Ad Post Permalink,TikTok Clicks (Destination) (SUM),Organic_ID
0,2026-07-02,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q2_AON_Soc_TikTok_Cons...,06804027_Other_Me-manche?_NA_76261405102472266...,TIKTOK_1864652638213121,06804027_Other_Me-manche?_NA_76261405102472266...,0,¿Vives mirándote al espejo cuando entrenas con...,https://prod.cdata.app.sprinklr.com/PAID/312/2...,...,0,0,5,1,3,0,20284361506,https://www.tiktok.com/@kotexcol/video/7626140...,0,7626140510247226645
1,2026-07-02,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q2_AON_Soc_TikTok_Cons...,06804027_Other_Colores_NA_7628383524583705877_...,TIKTOK_1864652727323713,06804027_Other_Colores_NA_7628383524583705877_...,0,¡Si! Los colores de tu periodo pueden cambiar ...,https://prod.cdata.app.sprinklr.com/PAID/312/0...,...,0,0,21,6,6,0,20284361531,https://www.tiktok.com/@kotexcol/video/7628383...,0,7628383524583705877
2,2026-07-02,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q2_AON_Soc_TikTok_Cons...,06804027_Other_Toalla-según-tu-flujo_NA_762873...,TIKTOK_1864652836586561,06804027_Other_Toalla-según-tu-flujo_NA_762873...,0,"¿No sabes si tu flujo menstrual es leve, norma...",https://prod.cdata.app.sprinklr.com/PAID/312/7...,...,0,0,4,13,5,0,20284361528,https://www.tiktok.com/@kotexcol/video/7628738...,0,7628738746585730325
3,2026-07-02,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q2_AON_Soc_TikTok_Cons...,06804027_Other_Explicado-con-gatitos_NA_763246...,TIKTOK_1864652926699601,06804027_Other_Explicado-con-gatitos_NA_763246...,0,Cómo entender a tu novia con el periodo 101. 🎀...,https://prod.cdata.app.sprinklr.com/PAID/312/3...,...,0,0,19,13,3,0,20316242306,https://www.tiktok.com/@kotexcol/video/7632465...,0,7632465389670157575
4,2026-07-02,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q2_AON_Soc_TikTok_Cons...,06804027_Other_Colores_NA_7628383524583705877_...,TIKTOK_1864653045834786,06804027_Other_Colores_NA_7628383524583705877_...,0,¡Si! Los colores de tu periodo pueden cambiar ...,https://prod.cdata.app.sprinklr.com/PAID/312/0...,...,0,0,21,6,6,0,20284361531,https://www.tiktok.com/@kotexcol/video/7628383...,0,7628383524583705877
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2531,2026-07-31,OMD_CO_ES_KC-BCC_HUGGIES,Instagram,EM_CO_BCC_Hugg_2026Q3_AON_Soc_Meta_Awa_Dermaca...,07099480_Other_ToyStory-Generaciones_NA_DZqvgK...,FACEBOOK_120247829342560719,07099480_Other_ToyStory-Generaciones_NA_DZqvgK...,Huggies® Colombia (@huggiesco) • Instagram pho...,La magia pasa de generación en generación y ho...,https://prod.cdata.app.sprinklr.com/PAID/312/9...,...,0,0,0,0,0,0,20863822444,https://www.instagram.com/p/DZqvgKknJnc/,0,DZqvgKknJnc
2532,2026-07-31,OMD_CO_ES_KC-BCC_HUGGIES,Instagram,EM_CO_BCC_Hugg_2026Q3_AON_Soc_Meta_Awa_Dermaca...,07099480_Other_TallaDelPañal_NA_DZgVw5dlF2b_NO...,FACEBOOK_120247829342570719,07099480_Other_TallaDelPañal_NA_DZgVw5dlF2b_NO...,0,¿Dudas con la talla del pañal? ¡El secreto est...,https://prod.cdata.app.sprinklr.com/PAID/312/2...,...,0,0,0,1,0,1,20823510874,https://www.instagram.com/reel/DZgVw5dlF2b/,0,DZgVw5dlF2b
2533,2026-07-31,OMD_CO_ES_KC-BCC_HUGGIES,Instagram,EM_CO_BCC_Hugg_2026Q3_AON_Soc_Meta_Awa_Dermaca...,07099480_Other_PaternidadReal_NA_DZdujfxjHLk_N...,FACEBOOK_120247829342580719,07099480_Other_PaternidadReal_NA_DZdujfxjHLk_N...,Huggies® Colombia (@huggiesco) • Instagram pho...,"Fresco, papá. Es normal sentir que el mundo te...",https://prod.cdata.app.sprinklr.com/PAID/312/9...,...,0,0,1,0,0,0,20811399537,https://www.instagram.com/p/DZdujfxjHLk/,0,DZdujfxjHLk
2534,2026-07-31,OMD_CO_ES_KC-BCC_HU

In [26]:
# Correct column 'Spent (USD) in USD (SUM)' format
sprinklr_paid_new['Spent (USD) in USD (SUM)'] = (
    sprinklr_paid_new['Spent (USD) in USD (SUM)']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

In [27]:
# Group values (sum) by Organic Id, Social Network, Date
sprinklr_paid_new_consolidated = sprinklr_paid_new.groupby(['Organic_ID','Date', 'Ad Account', 'Social Network', 'Paid Initiative Name', 'Ad Variant Name', 'Ad Variant Id', 'Ad Variant', 'Title',
    'Body', 'Image URL', 'Ad Post Id'], as_index=False)[[
    'Impressions (SUM)',
    'Spent (USD) in USD (SUM)',
    'TikTok Video views (SUM)',
    'TikTok 6-second video views (SUM)',
    'Facebook Video Plays (3 sec) (SUM)',
    'Facebook Video Plays to 25% (SUM)',
    'TikTok Paid comments (SUM)',
    'TikTok Paid shares (SUM)',
    'TikTok Paid likes (SUM)',
    'Facebook Post Comments (SUM)',
    'Facebook Post Shares (SUM)',
    'Facebook Post Likes (SUM)',
    'Facebook Link Clicks (SUM)',
    'Facebook Reactions (SUM)',
    'Facebook Post Saves (SUM)',
    'Neutral Sentiment Count (Paid + Organic) (AVG)',
    'Positive Sentiment Count (Paid + Organic) (AVG)',
    'Negative Sentiment Count (Paid + Organic) (AVG)',
    'Facebook Avg. Duration of Video Played (SUM)',
    'TikTok Clicks (Destination) (SUM)'
]].sum()

sprinklr_paid_new_consolidated

,Organic_ID,Date,Ad Account,Social Network,Paid Initiative Name,Ad Variant Name,Ad Variant Id,Ad Variant,Title,Body,...,Facebook Post Shares (SUM),Facebook Post Likes (SUM),Facebook Link Clicks (SUM),Facebook Reactions (SUM),Facebook Post Saves (SUM),Neutral Sentiment Count (Paid + Organic) (AVG),Positive Sentiment Count (Paid + Organic) (AVG),Negative Sentiment Count (Paid + Organic) (AVG),Facebook Avg. Duration of Video Played (SUM),TikTok Clicks (Destination) (SUM)
0,7626140510247226645,2026-07-02,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q2_AON_Soc_TikTok_Cons...,06804027_Other_Me-manche?_NA_76261405102472266...,TIKTOK_1864652638213121,06804027_Other_Me-manche?_NA_76261405102472266...,0,¿Vives mirándote al espejo cuando entrenas con...,...,0,0,0,0,0,5,1,3,0,0
1,7626140510247226645,2026-07-02,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q2_AON_Soc_TikTok_Cons...,06804027_Other_Me-manche?_NA_76261405102472266...,TIKTOK_1864653045836018,06804027_Other_Me-manche?_NA_76261405102472266...,0,¿Vives mirándote al espejo cuando entrenas con...,...,0,0,0,0,0,5,1,3,0,0
2,7626140510247226645,2026-07-02,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q2_AON_Soc_TikTok_Cons...,06804027_Other_Me-manche?_NA_76261405102472266...,TIKTOK_1864653045840146,06804027_Other_Me-manche?_NA_76261405102472266...,0,¿Vives mirándote al espejo cuando entrenas con...,...,0,0,0,0,0,5,1,3,0,0
3,7626140510247226645,2026-07-03,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q2_AON_Soc_TikTok_Cons...,06804027_Other_Me-manche?_NA_76261405102472266...,TIKTOK_1864653045836018,06804027_Other_Me-manche?_NA_76261405102472266...,0,¿Vives mirándote al espejo cuando entrenas con...,...,0,0,0,0,0,5,1,3,0,0
4,7628383524583705877,2026-07-02,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q2_AON_Soc_TikTok_Cons...,06804027_Other_Colores_NA_7628383524583705877_...,TIKTOK_1864652727323713,06804027_Other_Colores_NA_7628383524583705877_...,0,¡Si! Los colores de tu periodo pueden cambiar ...,...,0,0,0,0,0,21,6,6,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2451,DadpNGQmCaR,2026-07-29,OMD_AR_ES_KC-FEM_KOTEX,Instagram,EM_AR_FemCare_Kotex_2026Q3_AON_Soc_Meta_Awa_cu...,07095170_Other_estoy-jugada_Meli_DadpNGQmCaR_N...,FACEBOOK_120249701128620638,07095170_Other_estoy-jugada_Meli_DadpNGQmCaR_N...,Kotex,Cuando estás menstruando sentís que estás juga...,...,0,0,0,0,0,0,0,0,0,0
2452,DadpNGQmCaR,2026-07-30,OMD_AR_ES_KC-FEM_KOTEX,Instagram,EM_AR_FemCare_Kotex_2026Q3_AON_Soc_Meta_Awa_cu...,07095170_Other_estoy-jugada_Meli_DadpNGQmCaR_N...,FACEBOOK_120249701128620638,07095170_Other_estoy-jugada_Meli_DadpNGQmCaR_N...,Kotex,Cuando estás menstruando sentís que estás juga...,...,0,0,0,0,0,0,0,0,0,0
2453,DaeH9Y1xNhK,2026-07-15,OMD_PE_ES_KC-BCC_HUGGIES,Instagram,EM_PE_BCC_Hugg_2026Q3_AON_Soc_Meta_Awa_CLASSIC...,07166189_Other_mas-huggies-mas-cine_Inkfrma_Da...,FACEBOOK_120247134247300259,07166189_Other_mas-huggies-mas-cine_Inkfrma_Da...,Inkafarma: Más salud al mejor precio,"¡Llegó la promo Más Huggies, Más Cine! ❤️\n\nC...",...,0,0,0,0,0,0,0,0,0,0
2454,DaeH9Y1xNhK,2026-07-16,OMD_PE_ES_KC-BCC_HUGGIES,Instagram,EM_PE_BCC_Hugg_2026Q3_AON_Soc_Meta_Awa_CLASSIC...,07166189_Other_mas-huggies-mas-cine_Inkfrma_Da...,FACEBOOK_120247134247300259,07166189_Other_mas-huggies-mas-cine_Inkfrma_Da...,Inkafarma: Más salud al mejor precio,"¡Llegó la promo Más Huggies, Más Cine! ❤️\n\nC...",...,0,0,0,0,0,0,0,0,0,0


In [28]:
# Update historic paid file with new data
# Open historic paid file
historic_paid = gc.open_by_key('1giT7UA49YozfGI8kDNY-XYe609CA9hg3flzTINgfFjM')
historic_paid = historic_paid.worksheet('Hoja 1')
historic_paid = get_as_dataframe(historic_paid)

# Ensure 'Date' columns share the exact same data type (crucial for exact matching)
sprinklr_paid_new_consolidated['Date'] = pd.to_datetime(sprinklr_paid_new_consolidated['Date'])
historic_paid['Date'] = pd.to_datetime(historic_paid['Date'])

# Append new data and drop duplicates based on the composite key
historic_paid_updated = (
    pd.concat([historic_paid, sprinklr_paid_new_consolidated], ignore_index=True)
    .drop_duplicates(subset=['Date', 'Organic_ID', 'Paid Initiative Name', 'Ad Variant Id'], keep='first')
    .reset_index(drop=True)
)

In [29]:
# Save the consolidated database (historic+new) as the new historic file (replacing previous data with the new consolidation)
# Open the destination sheets file
sh = gc.open_by_key('1giT7UA49YozfGI8kDNY-XYe609CA9hg3flzTINgfFjM')
worksheet = sh.worksheet('Hoja 1')

# Replace old data with new data
set_with_dataframe(worksheet, historic_paid_updated)
print("DataFrame saved successfully!")

DataFrame saved successfully!


In [30]:
# Work with historic file

# Load historic file
sprinklr_paid_historic_file = gc.open_by_key('1giT7UA49YozfGI8kDNY-XYe609CA9hg3flzTINgfFjM')
# Select the specific worksheet (0 is the first tab)
sprinklr_paid_historic_file_worksheet = sprinklr_paid_historic_file.worksheet('Hoja 1')
# Load the data directly into a pandas DataFrame
sprinklr_paid_historic = get_as_dataframe(sprinklr_paid_historic_file_worksheet)

In [31]:
# Work with historic file

# Group values (sum) by Organic Id and Social Network (get the total for each boosted post)
sprinklr_paid_historic_grouped = sprinklr_paid_historic.groupby(['Organic_ID', 'Ad Account', 'Social Network', 'Paid Initiative Name', 'Ad Variant Name', 'Ad Variant Id', 'Ad Variant', 'Title',
    'Body', 'Image URL', 'Ad Post Id'], as_index=False)[[
    'Impressions (SUM)',
    'Spent (USD) in USD (SUM)',
    'TikTok Video views (SUM)',
    'TikTok 6-second video views (SUM)',
    'Facebook Video Plays (3 sec) (SUM)',
    'Facebook Video Plays to 25% (SUM)',
    'TikTok Paid comments (SUM)',
    'TikTok Paid shares (SUM)',
    'TikTok Paid likes (SUM)',
    'Facebook Post Comments (SUM)',
    'Facebook Post Shares (SUM)',
    'Facebook Post Likes (SUM)',
    'Facebook Link Clicks (SUM)',
    'Facebook Reactions (SUM)',
    'Facebook Post Saves (SUM)',
    'Neutral Sentiment Count (Paid + Organic) (AVG)',
    'Positive Sentiment Count (Paid + Organic) (AVG)',
    'Negative Sentiment Count (Paid + Organic) (AVG)',
    'Facebook Avg. Duration of Video Played (SUM)',
    'TikTok Clicks (Destination) (SUM)'
]].sum()

In [32]:
# Work with historic file

# Create metrics
sprinklr_paid_historic_grouped['Qualified Paid Views'] = sprinklr_paid_historic_grouped['Facebook Video Plays to 25% (SUM)'] + sprinklr_paid_historic_grouped['TikTok 6-second video views (SUM)']

sprinklr_paid_historic_grouped['TikTok Paid Engagement'] = (
    sprinklr_paid_historic_grouped['TikTok Paid comments (SUM)'] +
    sprinklr_paid_historic_grouped['TikTok Paid shares (SUM)'] +
    sprinklr_paid_historic_grouped['TikTok Paid likes (SUM)']
)

sprinklr_paid_historic_grouped['Meta Paid Engagement'] = (
    sprinklr_paid_historic_grouped['Facebook Post Comments (SUM)'] +
    sprinklr_paid_historic_grouped['Facebook Post Shares (SUM)'] +
    sprinklr_paid_historic_grouped['Facebook Post Likes (SUM)'] +
    sprinklr_paid_historic_grouped['Facebook Post Saves (SUM)']
)

sprinklr_paid_historic_grouped['TikTok Paid Qualified Eng. Rate'] = sprinklr_paid_historic_grouped.apply(
    lambda row: row['TikTok Paid Engagement'] / row['Qualified Paid Views']
    if row['Qualified Paid Views'] != 0 else 0, axis=1
)

sprinklr_paid_historic_grouped['Meta Paid Qualified Eng. Rate'] = sprinklr_paid_historic_grouped.apply(
    lambda row: row['Meta Paid Engagement'] / row['Qualified Paid Views']
    if row['Qualified Paid Views'] != 0 else 0, axis=1
)

sprinklr_paid_historic_grouped['CPI'] = sprinklr_paid_historic_grouped.apply(
    lambda row: row['Spent (USD) in USD (SUM)'] / row['Impressions (SUM)']
    if row['Impressions (SUM)'] != 0 else 0, axis=1
)

sprinklr_paid_historic_grouped['CPC'] = sprinklr_paid_historic_grouped.apply(
    lambda row: row['Spent (USD) in USD (SUM)'] / (row['Facebook Link Clicks (SUM)'] + row['TikTok Clicks (Destination) (SUM)'] )
    if row['Facebook Link Clicks (SUM)'] != 0 else 0, axis=1
)



In [33]:
# Save sprinklr_paid_final_table
# Open the destination sheets file
sh = gc.open_by_key('1W73RHKRuKfp-AAVQDgrMSwP3huq0r8-bDeRMLjbPxZA')
worksheet = sh.worksheet('Hoja 1')
# Replace old data with new data
set_with_dataframe(worksheet, sprinklr_paid_historic_grouped)
print("DataFrame saved successfully!")

DataFrame saved successfully!
